# Burst 1 (~25 CUs on NVIDIA L4 24GB): Week 1 CUDA Closure — 4-Bit `bitsandbytes` NF4 QLoRA vs. `llm-compressor` W4A16 Serving Quantization

**Sprint**: Applied AI Foundry (v4.0) | **Target Repo**: [`imhlg/qwen3-4b-finops-w4a16-vllm`](https://huggingface.co/imhlg/qwen3-4b-finops-w4a16-vllm)  
**Venture Context**: [`curons.org`](https://curons.org) B2B LLM FinOps & Weight Optimization Foundry

### Architectural Objective (Why Both Quantization Formats Exist)
1. **Training Quantization (`bitsandbytes` 4-Bit NF4 + Double Quantization)**: Compresses frozen base weights during **backpropagation** so 4B–70B models fine-tune inside minimal VRAM with a LoRA adapter attached in `bfloat16`/`float16`.
2. **Serving Quantization (`llm-compressor` W4A16 GPTQ/RTN)**: After merging LoRA back into BF16 (`merge_and_unload()`), compresses linear weights to 4-bit integer groups unpacked inside **`vLLM` Marlin CUDA kernels** during Tensor Core GEMM—giving ~70% static VRAM savings *and* high-concurrency throughput.

> **Colab Runtime**: Select **Runtime → Change runtime type → L4 GPU** (or A100), add `HF_TOKEN` (Write permission) to **Colab Secrets** (key icon on left sidebar), then click **Runtime → Run all**. Disconnect runtime as soon as Cell 5 finishes pushing to Hugging Face!

In [ ]:
%pip install -q -U "transformers>=5.0.0" "trl>=0.15.0" "peft>=0.15.0" "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "datasets>=3.0.0" "huggingface_hub>=0.28.0" "llmcompressor"

import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not detected! Go to Runtime -> Change runtime type and select L4 or A100 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16

print(f"GPU Detected       : {gpu_name} (Compute Capability SM {major}.{minor})")
print(f"Total GPU VRAM     : {vram_gb:.2f} GB")
print(f"Selected Precision : {compute_dtype}")

In [ ]:
import os
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Authenticated to Hugging Face Hub via Colab Secret HF_TOKEN.")
else:
    print("HF_TOKEN secret not found. Launching interactive notebook login...")
    notebook_login()

In [ ]:
# ==============================================================================
# PART 1: 4-BIT BITSANDBYTES NF4 QLORA TRAINING (TRL SFTTrainer)
# ==============================================================================
import gc
import inspect
import json
from pathlib import Path
import time

from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen3-4B"
HUB_REPO_ID = "imhlg/qwen3-4b-finops-w4a16-vllm"
ADAPTER_DIR = "outputs/qwen3-4b-nf4-qlora"
MERGED_DIR = "outputs/qwen3-4b-merged-bf16"
W4A16_DIR = "outputs/qwen3-4b-finops-w4a16-vllm"

Path("outputs").mkdir(parents=True, exist_ok=True)


def build_finops_dataset(tokenizer: AutoTokenizer, repeats: int = 8) -> Dataset:
    """Create a 64-example B2B FinOps & GPU Inference Architecture SFT dataset."""
    scenarios = [
        (
            "Our AWS g5.2xlarge (24GB A10G) running a 7B FP16 model hits CUDA OOM at 16 concurrent users with 4,096 context length. Diagnose the bottleneck and prescribe the fix.",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: 7B FP16 weights consume 14.0 GB VRAM, leaving ~8.5 GB for KV-cache and activations. At 16 x 4,096 tokens, contiguous FP16 KV-cache allocation triggers OOM.\n"
            "2. Remediation: Quantize weights via llm-compressor to W4A16 (~4.2 GB) and serve on vLLM with PagedAttention + FP8 KV-cache.\n"
            "3. Projected ROI: Cuts weight VRAM by 70%, scales concurrency from 16 to 96+ users on one A10G, reducing unit token cost by 78%.",
        ),
        (
            "Why is our batch inference pipeline emitting repetitive garbage tokens when batch_size > 1 on a decoder-only Qwen/Llama model?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Default right-padding (`padding_side='right'`) places <pad> tokens at index -1 for shorter prompts, corrupting causal next-token sampling.\n"
            "2. Remediation: Enforce `tokenizer.padding_side = 'left'` and supply explicit `attention_mask` tensors so left-padded positions are masked to -inf.\n"
            "3. Projected ROI: Eliminates 100% of malformed batch outputs and unlocks high-throughput continuous batching.",
        ),
        (
            "We have 12 enterprise product teams each wanting a custom fine-tuned 4B model. Do we need 12 separate GPU instances?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Hosting 12 full-weight checkpoints duplicates 12x static base weights across separate GPU nodes.\n"
            "2. Remediation: Train rank-8 LoRA adapters (~35-65 MB each) and serve one shared W4A16 base checkpoint in vLLM using `--enable-lora`.\n"
            "3. Projected ROI: Consolidates 12 GPU instances into 1 shared HA pair, cutting monthly GPU spend by 83%.",
        ),
        (
            "Our JSON extraction endpoint has a 1,800ms TTFT on Qwen3-4B. How do we push TTFT under 250ms?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Hybrid reasoning (`enable_thinking=True`) burns hundreds of latent `<think>...</think>` tokens prior to emitting the opening JSON brace.\n"
            "2. Remediation: Set `enable_thinking=False` in `apply_chat_template` and enforce JSON syntax at the logit layer via Outlines FSM masking.\n"
            "3. Projected ROI: Drops TTFT from 1,800ms to <190ms and cuts generated output token spend by 65%.",
        ),
        (
            "Should we serve a bitsandbytes 4-bit NF4 checkpoint directly behind our high-concurrency production FastAPI gateway?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: bitsandbytes NF4 uses a non-uniform lookup table designed for low-batch QLoRA backprop, creating dequantization bottlenecks above batch size 8.\n"
            "2. Remediation: Merge LoRA deltas back to BF16 (`merge_and_unload()`) and quantize with `llm-compressor` to W4A16 (Marlin kernel) or FP8 for vLLM.\n"
            "3. Projected ROI: Delivers 2.4x higher batch throughput at identical 4-bit VRAM footprint on NVIDIA L4 GPUs.",
        ),
        (
            "How can an enterprise run an OLMoE-1B-7B Mixture-of-Experts model inside a single 16GB/24GB GPU with high KV-cache headroom?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Although only 1.3B parameters activate per token, all 64 experts (~13.8 GB BF16) must remain resident in HBM VRAM.\n"
            "2. Remediation: Profile domain routing frequency and Gate-Weighted MSAN, excise unused experts (<2%), and run a 1-epoch TRL distillation pass.\n"
            "3. Projected ROI: Frees ~4.3 GB static VRAM (-31%), doubling maximum concurrent KV-cache slots on a single L4 GPU.",
        ),
        (
            "Our external Llama-Guard LLM judge adds 550ms latency and doubles GPU costs on every customer turn. What is the alternative?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Serial external judge calls pay full prefill/decode overhead twice per user turn.\n"
            "2. Remediation: Attach PyTorch forward hooks on middle-layer residual streams (Layer 18), score with a 10 KB linear probe, and clamp via a CAA steering vector.\n"
            "3. Projected ROI: Cuts guardrail overhead from 550ms to <2ms and reduces guardrail compute spend by 99%.",
        ),
        (
            "How do we satisfy a GDPR deletion request for 100 memorized PII records inside a 7B model without retraining from scratch?",
            "[FINOPS_AUDIT_REPORT]\n"
            "1. Root Cause: Raw gradient ascent destabilizes weights toward -infinity; full retraining wastes thousands of dollars in GPU compute.\n"
            "2. Remediation: Apply Negative Preference Optimization (NPO) in TRL anchored to a frozen reference model plus retain-set SFT regularization.\n"
            "3. Projected ROI: Achieves 0% PII extraction recall in <30 minutes on one A100 while preserving >99% schema accuracy.",
        ),
    ]

    records = []
    system_prompt = (
        "You are the Lead AI Systems & FinOps Architect at curons.org. "
        "Always respond using the structured [FINOPS_AUDIT_REPORT] format with "
        "1. Root Cause, 2. Remediation, and 3. Projected ROI."
    )
    for _ in range(repeats):
        for q, a in scenarios:
            msgs = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": q},
                {"role": "assistant", "content": a},
            ]
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, enable_thinking=False
            )
            records.append({"text": text})
    return Dataset.from_list(records)


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

torch.cuda.reset_peak_memory_stats()
t_load = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
load_time_s = time.perf_counter() - t_load
nf4_load_vram_gb = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"NF4 Base Loaded in {load_time_s:.2f}s | VRAM Allocated: {nf4_load_vram_gb:.2f} GB")

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
peft_model = get_peft_model(model, lora_config)
trainable_params, all_params = peft_model.get_nb_trainable_parameters()
print(f"QLoRA Trainable Params: {trainable_params:,} / {all_params:,} ({100*trainable_params/all_params:.4f}%)")

train_ds = build_finops_dataset(tokenizer, repeats=8)
sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = dict(
    output_dir=ADAPTER_DIR,
    max_steps=30,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    report_to="none",
    seed=42,
)
if "dataset_text_field" in sft_params:
    sft_kwargs["dataset_text_field"] = "text"
if "max_length" in sft_params:
    sft_kwargs["max_length"] = 384
elif "max_seq_length" in sft_params:
    sft_kwargs["max_seq_length"] = 384

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_ds,
    args=SFTConfig(**sft_kwargs),
    processing_class=tokenizer,
)

torch.cuda.synchronize()
t0 = time.perf_counter()
train_result = trainer.train()
torch.cuda.synchronize()
qlora_train_s = time.perf_counter() - t0
qlora_peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

loss_logs = [x["loss"] for x in trainer.state.log_history if "loss" in x]
initial_loss = loss_logs[0] if loss_logs else float(train_result.training_loss)
final_loss = loss_logs[-1] if loss_logs else float(train_result.training_loss)

print(
    f"QLoRA 30-step SFT done in {qlora_train_s:.1f}s | Peak VRAM: {qlora_peak_vram_gb:.2f} GB "
    f"| Loss: {initial_loss:.4f} -> {final_loss:.4f}"
)

peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

del trainer, peft_model, model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# PART 2: MERGE ADAPTER TO BF16 & QUANTIZE WITH LLM-COMPRESSOR (W4A16)
# ==============================================================================
from peft import PeftModel
from transformers import DynamicCache

print("Reloading base model in", compute_dtype, "to merge QLoRA delta into unified weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map="auto",
    attn_implementation="sdpa",
)
merged_peft = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_peft.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

merged_size_gb = sum(
    f.stat().st_size for f in Path(MERGED_DIR).rglob("*.safetensors")
) / (1024 ** 3)
print(f"Merged BF16/FP16 checkpoint saved to {MERGED_DIR} ({merged_size_gb:.2f} GB)")

del merged_model, merged_peft, base_model
gc.collect()
torch.cuda.empty_cache()

# Build tokenized calibration dataset for llm-compressor
calib_raw = build_finops_dataset(tokenizer, repeats=4).select(range(32))

def _tok_calib(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=384,
        padding=False,
    )

calib_tokenized = calib_raw.map(_tok_calib, batched=True, remove_columns=["text"])

quant_method = "llm-compressor GPTQ W4A16"
try:
    from llmcompressor import oneshot
    from llmcompressor.modifiers.quantization import GPTQModifier

    recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
    t_q = time.perf_counter()
    oneshot(
        model=MERGED_DIR,
        dataset=calib_tokenized,
        recipe=recipe,
        output_dir=W4A16_DIR,
        max_seq_length=384,
        num_calibration_samples=32,
    )
    quant_time_s = time.perf_counter() - t_q
except Exception as gptq_err:
    print(f"GPTQModifier path raised ({gptq_err}); falling back to QuantizationModifier RTN W4A16...")
    from llmcompressor import oneshot
    from llmcompressor.modifiers.quantization import QuantizationModifier

    quant_method = "llm-compressor RTN W4A16"
    recipe = QuantizationModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
    t_q = time.perf_counter()
    oneshot(
        model=MERGED_DIR,
        recipe=recipe,
        output_dir=W4A16_DIR,
    )
    quant_time_s = time.perf_counter() - t_q

tokenizer.save_pretrained(W4A16_DIR)
w4a16_size_gb = sum(
    f.stat().st_size for f in Path(W4A16_DIR).rglob("*.safetensors")
) / (1024 ** 3)
compression_pct = (1.0 - (w4a16_size_gb / max(merged_size_gb, 1e-6))) * 100.0
print(
    f"{quant_method} complete in {quant_time_s:.1f}s | "
    f"Disk Size: {merged_size_gb:.2f} GB -> {w4a16_size_gb:.2f} GB (-{compression_pct:.1f}%)"
)

# Verify inference & CUDA throughput on the compressed checkpoint
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

eval_model = AutoModelForCausalLM.from_pretrained(
    W4A16_DIR,
    device_map="auto",
)
eval_model.eval()

test_prompt = (
    "A client spends $40,000/month running unquantized 7B FP16 endpoints on AWS g5.2xlarge nodes "
    "at 16 concurrent requests. Give a 3-point FinOps optimization prescription."
)
eval_inputs = tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "You are the Lead AI Systems & FinOps Architect at curons.org. Always respond using the structured [FINOPS_AUDIT_REPORT] format with 1. Root Cause, 2. Remediation, and 3. Projected ROI.",
        },
        {"role": "user", "content": test_prompt},
    ],
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to(eval_model.device)

kv_cache = DynamicCache()
torch.cuda.synchronize()
t_gen = time.perf_counter()
with torch.inference_mode():
    gen_ids = eval_model.generate(
        **eval_inputs,
        past_key_values=kv_cache,
        max_new_tokens=160,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
torch.cuda.synchronize()
gen_elapsed_s = time.perf_counter() - t_gen
new_tokens = gen_ids.shape[-1] - eval_inputs["input_ids"].shape[-1]
cuda_tok_per_sec = new_tokens / max(gen_elapsed_s, 1e-6)
serving_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
sample_output = tokenizer.decode(
    gen_ids[0, eval_inputs["input_ids"].shape[-1]:], skip_special_tokens=True
)

print(
    f"CUDA Eval: {new_tokens} tokens in {gen_elapsed_s:.2f}s ({cuda_tok_per_sec:.1f} tok/s) "
    f"| Peak Serving VRAM: {serving_vram_gb:.2f} GB"
)
print("\n--- Sample Output ---\n" + sample_output.strip())

In [ ]:
# ==============================================================================
# PART 3: GENERATE TELEMETRY MODEL CARD & PUSH TO HUGGING FACE HUB (imhlg)
# ==============================================================================
from huggingface_hub import HfApi

telemetry = {
    "model_id": MODEL_ID,
    "hub_repo_id": HUB_REPO_ID,
    "gpu_name": gpu_name,
    "compute_capability": f"SM {major}.{minor}",
    "compute_dtype": str(compute_dtype),
    "nf4_load_vram_gb": round(nf4_load_vram_gb, 3),
    "qlora_peak_vram_gb": round(qlora_peak_vram_gb, 3),
    "qlora_train_seconds_30steps": round(qlora_train_s, 2),
    "initial_loss": round(initial_loss, 4),
    "final_loss": round(final_loss, 4),
    "merged_checkpoint_gb": round(merged_size_gb, 3),
    "quant_method": quant_method,
    "w4a16_checkpoint_gb": round(w4a16_size_gb, 3),
    "disk_reduction_pct": round(compression_pct, 2),
    "cuda_eval_tok_per_sec": round(cuda_tok_per_sec, 2),
    "cuda_serving_peak_vram_gb": round(serving_vram_gb, 3),
}

with open(Path(W4A16_DIR) / "telemetry.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

readme = f"""---
base_model: {MODEL_ID}
tags:
  - qlora
  - bitsandbytes
  - nf4
  - llm-compressor
  - w4a16
  - vllm
  - finops
  - curons
  - applied-ai-foundry
license: apache-2.0
---

# {HUB_REPO_ID.split('/')[-1]}: 4-Bit NF4 QLoRA + `llm-compressor` W4A16 Serving Checkpoint

Built as **Burst 1 (Week 1 CUDA Closure)** of the **Applied AI Foundry (`v4.0`)** Sprint (**[`curons.org`](https://curons.org)** | **[`@imhlg`](https://huggingface.co/imhlg)**).

Demonstrates the production post-training handoff:
1. **Training Phase**: 4-bit `bitsandbytes` NF4 + Double Quantization QLoRA (`r=8, alpha=16`) via `TRL` `SFTTrainer` (`{initial_loss:.4f} -> {final_loss:.4f}` loss across 30 steps in `{qlora_train_s:.1f}s`).
2. **Serving Phase**: Merged LoRA deltas back into `{compute_dtype}` (`merge_and_unload()`, `{merged_size_gb:.2f} GB`) and compressed via **`{quant_method}`** down to **`{w4a16_size_gb:.2f} GB`** (`-{compression_pct:.1f}%` footprint) for `vLLM` Marlin CUDA serving.

## Hardware Telemetry Matrix (Apple M4 MPS vs. Colab NVIDIA CUDA)

| Stage / Backend | Precision / Format | Peak Memory | Training / Eval Speed |
| :--- | :--- | :--- | :--- |
| **Apple M4 Baseline (Week 1)** | `bfloat16` (`mps`) | `7.50 GB` Metal driver | `10.5 tok/s` (15-step SFT: `95.8s`) |
| **Colab CUDA QLoRA (`{gpu_name}`)** | `NF4` 4-bit + Double Quant | `{qlora_peak_vram_gb:.2f} GB` peak CUDA | `30 steps` in `{qlora_train_s:.1f}s` |
| **Merged BF16 Checkpoint** | `{compute_dtype}` | `{merged_size_gb:.2f} GB` safetensors | — |
| **Production `{quant_method}`** | `W4A16` (4-bit weights, 16-bit acts) | **`{w4a16_size_gb:.2f} GB`** disk / `{serving_vram_gb:.2f} GB` peak | **`{cuda_tok_per_sec:.1f} tok/s`** |
"""

with open(Path(W4A16_DIR) / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

api = HfApi()
api.create_repo(repo_id=HUB_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=W4A16_DIR,
    repo_id=HUB_REPO_ID,
    repo_type="model",
    commit_message="feat: publish Burst 1 NF4 QLoRA + llm-compressor W4A16 checkpoint & CUDA telemetry",
)

print(f"SUCCESS! Published to https://huggingface.co/{HUB_REPO_ID}")
print("ACTION REQUIRED: Click 'Runtime -> Disconnect and delete runtime' NOW to stop Colab CU burn.")